In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SETUP — rode esta célula primeiro
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, sys

pkgs = ["matplotlib", "seaborn", "sqlalchemy", "psycopg2-binary", "python-dotenv", "numpy"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], capture_output=True)

print("✅ Dependências instaladas — agora vá em  Kernel → Restart  e rode as próximas células")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os, warnings
warnings.filterwarnings('ignore')

load_dotenv('../config/.env')
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{quote_plus(os.getenv('DB_PASSWORD'))}"
    f"@localhost:5432/{os.getenv('DB_NAME')}"
)

df = pd.read_sql("SELECT * FROM uirauna_weather ORDER BY datetime ASC", con=engine)
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('America/Fortaleza')
df['hora'] = df['datetime'].dt.hour
df['dia']  = df['datetime'].dt.date

print(f"✅ {len(df)} registros · {df['datetime'].min().strftime('%d/%m/%Y %H:%M')} → {df['datetime'].max().strftime('%d/%m/%Y %H:%M')}")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DASHBOARD PRINCIPAL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ultimo = df.iloc[-1]

BG      = '#0d1117'
BG2     = '#161b22'
BORDER  = '#30363d'
GREEN   = '#3fb950'
BLUE    = '#58a6ff'
ORANGE  = '#f0883e'
RED     = '#ff7b72'
PURPLE  = '#bc8cff'
YELLOW  = '#e3b341'
WHITE   = '#e6edf3'
GRAY    = '#8b949e'

fig = plt.figure(figsize=(18, 14), facecolor=BG)
gs  = gridspec.GridSpec(4, 4, figure=fig, hspace=0.45, wspace=0.35)

# ── helpers ────────────────────────────────────────────────
def card(ax, valor, titulo, cor, unidade=''):
    ax.set_facecolor(BG2)
    for sp in ax.spines.values():
        sp.set_edgecolor(cor); sp.set_linewidth(2)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.62, f"{valor}{unidade}", ha='center', va='center',
            fontsize=26, fontweight='bold', color=cor, transform=ax.transAxes)
    ax.text(0.5, 0.22, titulo, ha='center', va='center',
            fontsize=10, color=GRAY, transform=ax.transAxes)

def style_ax(ax, title='', xlabel='', ylabel='', cor=WHITE):
    ax.set_facecolor(BG2)
    ax.tick_params(colors=GRAY, labelsize=8)
    for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
    ax.title.set_color(cor); ax.title.set_fontsize(11); ax.title.set_fontweight('bold')
    if title:   ax.set_title(title)
    if xlabel:  ax.set_xlabel(xlabel, color=GRAY, fontsize=8)
    if ylabel:  ax.set_ylabel(ylabel, color=GRAY, fontsize=8)
    ax.xaxis.label.set_color(GRAY)
    ax.yaxis.label.set_color(GRAY)
    ax.tick_params(axis='x', rotation=25)

# ── TÍTULO ─────────────────────────────────────────────────
ax_title = fig.add_subplot(gs[0, :])
ax_title.set_facecolor(BG)
ax_title.axis('off')
ax_title.text(0.5, 0.75, '🌤  Clima de Uiraúna / PB',
              ha='center', fontsize=22, fontweight='bold', color=WHITE, transform=ax_title.transAxes)
ax_title.text(0.5, 0.15, f"Última atualização: {ultimo['datetime'].strftime('%d/%m/%Y às %H:%M')}  ·  Condição: {ultimo['weather_description'].title()}",
              ha='center', fontsize=11, color=GRAY, transform=ax_title.transAxes)

# ── CARDS ──────────────────────────────────────────────────
ax_c1 = fig.add_subplot(gs[1, 0])
ax_c2 = fig.add_subplot(gs[1, 1])
ax_c3 = fig.add_subplot(gs[1, 2])
ax_c4 = fig.add_subplot(gs[1, 3])

card(ax_c1, f"{ultimo['temperature']:.1f}", '🌡️  Temperatura',  RED,    ' °C')
card(ax_c2, f"{ultimo['humidity']:.0f}",    '💧  Umidade',      BLUE,   ' %')
card(ax_c3, f"{ultimo['wind_speed']:.1f}",  '💨  Vento',        GREEN,  ' m/s')
card(ax_c4, f"{ultimo['clouds']:.0f}",      '☁️  Nebulosidade', PURPLE, ' %')

# ── TEMPERATURA ────────────────────────────────────────────
ax_temp = fig.add_subplot(gs[2, :2])
ax_temp.set_facecolor(BG2)
ax_temp.fill_between(df['datetime'], df['temp_min'], df['temp_max'], alpha=0.15, color=RED)
ax_temp.plot(df['datetime'], df['temperature'], color=RED,    linewidth=2.5, marker='o', markersize=5, label='Temperatura')
ax_temp.plot(df['datetime'], df['feels_like'],  color=ORANGE, linewidth=1.5, linestyle='--', marker='s', markersize=3, label='Sensação')
ax_temp.legend(facecolor=BG2, edgecolor=BORDER, labelcolor=WHITE, fontsize=8)
ax_temp.xaxis.set_major_formatter(mdates.DateFormatter('%Hh'))
style_ax(ax_temp, title='🌡️  Temperatura (°C)', ylabel='°C')

# ── UMIDADE ────────────────────────────────────────────────
ax_hum = fig.add_subplot(gs[2, 2:])
ax_hum.set_facecolor(BG2)
ax_hum.fill_between(df['datetime'], df['humidity'], alpha=0.25, color=BLUE)
ax_hum.plot(df['datetime'], df['humidity'], color=BLUE, linewidth=2.5, marker='o', markersize=5)
ax_hum.axhline(df['humidity'].mean(), color=YELLOW, linestyle='--', linewidth=1,
               label=f"Média {df['humidity'].mean():.0f}%")
ax_hum.legend(facecolor=BG2, edgecolor=BORDER, labelcolor=WHITE, fontsize=8)
ax_hum.xaxis.set_major_formatter(mdates.DateFormatter('%Hh'))
style_ax(ax_hum, title='💧  Umidade (%)', ylabel='%')

# ── VENTO ──────────────────────────────────────────────────
ax_wind = fig.add_subplot(gs[3, :2])
ax_wind.set_facecolor(BG2)
ax_wind.fill_between(df['datetime'], 0, df['wind_speed'], alpha=0.25, color=GREEN)
ax_wind.plot(df['datetime'], df['wind_speed'], color=GREEN, linewidth=2.5, marker='o', markersize=5, label='Velocidade')
if 'wind_gust' in df.columns and df['wind_gust'].notna().any():
    ax_wind.plot(df['datetime'], df['wind_gust'], color=YELLOW, linewidth=1.5,
                 linestyle='--', marker='s', markersize=3, label='Rajada')
ax_wind.legend(facecolor=BG2, edgecolor=BORDER, labelcolor=WHITE, fontsize=8)
ax_wind.xaxis.set_major_formatter(mdates.DateFormatter('%Hh'))
style_ax(ax_wind, title='💨  Vento (m/s)', ylabel='m/s')

# ── PRESSÃO ────────────────────────────────────────────────
ax_pres = fig.add_subplot(gs[3, 2:])
ax_pres.set_facecolor(BG2)
ax_pres.fill_between(df['datetime'], df['pressure'], df['pressure'].min()-1, alpha=0.25, color=PURPLE)
ax_pres.plot(df['datetime'], df['pressure'], color=PURPLE, linewidth=2.5, marker='o', markersize=5)
ax_pres.xaxis.set_major_formatter(mdates.DateFormatter('%Hh'))
style_ax(ax_pres, title='🔵  Pressão Atmosférica (hPa)', ylabel='hPa')

plt.savefig('dashboard_clima.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ Dashboard salvo em: dashboard_clima.png")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  MAPA DE CORRELAÇÃO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
cols = ['temperature','feels_like','humidity','pressure','wind_speed','clouds']
labels = ['Temp.','Sensação','Umidade','Pressão','Vento','Nuvens']
corr = df[cols].corr()

fig, ax = plt.subplots(figsize=(9, 7), facecolor=BG)
ax.set_facecolor(BG2)

cmap = LinearSegmentedColormap.from_list('custom', ['#1a3a5c', BG2, '#5c1a1a'])
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=1, linecolor=BG,
            ax=ax, annot_kws={'size': 12, 'color': WHITE},
            xticklabels=labels, yticklabels=labels)

ax.tick_params(colors=GRAY, labelsize=10)
ax.set_title('🔗  Correlação entre Variáveis Meteorológicas',
             color=WHITE, fontsize=13, fontweight='bold', pad=15)

for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
plt.tight_layout()
plt.savefig('correlacao_clima.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PADRÕES POR HORA DO DIA
#  (Melhor com 24+ registros)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
if len(df) < 6:
    print(f"⏳ Poucos dados ainda ({len(df)} registros). Volte quando tiver mais horas coletadas!")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG)
    variaveis = [
        ('temperature', '🌡️  Temperatura (°C)', RED),
        ('humidity',    '💧  Umidade (%)',       BLUE),
        ('wind_speed',  '💨  Vento (m/s)',       GREEN),
    ]
    for ax, (col, titulo, cor) in zip(axes, variaveis):
        media = df.groupby('hora')[col].mean()
        bars  = ax.bar(media.index, media.values, color=cor, alpha=0.85, width=0.7)
        ax.set_facecolor(BG2)
        ax.set_title(titulo, color=WHITE, fontsize=11, fontweight='bold')
        ax.set_xlabel('Hora do dia', color=GRAY, fontsize=9)
        ax.tick_params(colors=GRAY)
        for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
        # destaque da hora mais alta
        idx_max = media.values.argmax()
        bars[idx_max].set_edgecolor(YELLOW)
        bars[idx_max].set_linewidth(2.5)

    plt.suptitle('⏰  Médias por Hora do Dia — Uiraúna/PB',
                 fontsize=14, fontweight='bold', color=WHITE, y=1.02)
    plt.tight_layout()
    plt.savefig('horarios_clima.png', dpi=150, bbox_inches='tight', facecolor=BG)
    plt.show()
